# 01 — What Is LLM Routing? — Practical Examples

**Covers**: RouterResponse dataclass, naive vs routed comparison, routing decision logger, failure type catalog

In [ ]:
import os, time, json
from openai import OpenAI
from dataclasses import dataclass
from typing import Optional
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. RouterResponse — Normalized Return Type

In [ ]:
@dataclass
class RouterResponse:
    content: str
    model_used: str
    provider: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    fallback_used: bool = False
    attempt_count: int = 1
    error: Optional[str] = None

    @property
    def cost_usd(self) -> float:
        PRICING = {
            'gpt-4o':      (5.00e-6, 15.00e-6),
            'gpt-4o-mini': (0.15e-6,  0.60e-6),
        }
        pi, po = PRICING.get(self.model_used, (1e-6, 3e-6))
        return self.input_tokens * pi + self.output_tokens * po

    def summary(self) -> str:
        return (f"model={self.model_used} | {self.latency_ms:.0f}ms | "
                f"in={self.input_tokens} out={self.output_tokens} | "
                f"${self.cost_usd:.6f} | fallback={self.fallback_used}")

print('RouterResponse defined ✅')

## 2. Naive Approach vs Router Approach

In [ ]:
REQUESTS = [
    {'text': 'What is 2+2?',                        'complexity': 'simple'},
    {'text': 'Explain quantum entanglement briefly.',  'complexity': 'moderate'},
    {'text': 'Write a full binary search tree in Python with insert, delete, and search.', 'complexity': 'complex'},
]

# ❌ Naive: same model for everything
print('=== NAIVE: gpt-4o for all requests ===')
total_cost_naive = 0
for req in REQUESTS:
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': req['text']}],
        max_tokens=100
    )
    lat = (time.perf_counter() - t0) * 1000
    cost = r.usage.total_tokens * 5e-6  # input price approx
    total_cost_naive += cost
    print(f"  [{req['complexity']:>8}] {lat:.0f}ms ${cost:.5f}: {req['text'][:45]}")

print(f"  Total (naive): ${total_cost_naive:.5f}")
print()

# ✅ Smart: choose model by complexity
print('=== SMART: route by complexity ===')
ROUTE_MAP = {'simple': 'gpt-4o-mini', 'moderate': 'gpt-4o-mini', 'complex': 'gpt-4o'}
total_cost_smart = 0
for req in REQUESTS:
    model_id = ROUTE_MAP[req['complexity']]
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=model_id,
        messages=[{'role': 'user', 'content': req['text']}],
        max_tokens=100
    )
    lat = (time.perf_counter() - t0) * 1000
    price = 5e-6 if model_id == 'gpt-4o' else 0.15e-6
    cost = r.usage.total_tokens * price
    total_cost_smart += cost
    print(f"  [{req['complexity']:>8}] {model_id:<12} {lat:.0f}ms ${cost:.5f}: {req['text'][:35]}")

print(f"  Total (smart):  ${total_cost_smart:.5f}")
print(f"  Savings: {(1 - total_cost_smart/total_cost_naive)*100:.0f}%")

## 3. Routing Decision Logger

In [ ]:
import tiktoken
enc = tiktoken.encoding_for_model('gpt-4o-mini')

def log_routing_decision(request: str, chosen_model: str, reason: str) -> None:
    tokens = len(enc.encode(request))
    print(f"[ROUTER] {chosen_model:<22} reason='{reason}' tokens={tokens}")
    print(f"         request='{request[:60]}'")

# Simulate routing decisions
test_inputs = [
    ('Is the sky blue? Yes or no.',                         'gpt-4o-mini', 'simple lookup, <10 tokens expected'),
    ('Analyze the pros and cons of microservices vs monolith.', 'gpt-4o',  'complex analysis task'),
    ('Translate: Bonjour le monde!',                        'gpt-4o-mini', 'translation, mini handles it well'),
    ('Write a recursive quicksort with full test coverage.', 'gpt-4o',     'complex coding task'),
]

print('Routing decisions logged:')
for req, model, reason in test_inputs:
    log_routing_decision(req, model, reason)

## 4. Failure Types Catalog

In [ ]:
from openai import RateLimitError, APIStatusError, APITimeoutError

FAILURE_HANDLERS = {
    RateLimitError:   ('rate_limit',    'retry after backoff or switch provider'),
    APITimeoutError:  ('timeout',       'retry or route to faster model'),
    APIStatusError:   ('server_error',  'check status code; 500/503 → fallback provider'),
}

def classify_error(exc: Exception) -> tuple[str, str]:
    for exc_type, (name, action) in FAILURE_HANDLERS.items():
        if isinstance(exc, exc_type): return name, action
    return 'unknown', 'log and investigate'

# Simulate error handling
errors_to_handle = [
    RateLimitError('Rate limit exceeded', response=None, body={}),
]
for err in errors_to_handle:
    name, action = classify_error(err)
    print(f'Error type: {name}')  
    print(f'Recommended action: {action}')  

# General pattern
print('\nError classification pattern:')
for exc_type, (name, action) in FAILURE_HANDLERS.items():
    print(f'  {exc_type.__name__:<25} → {name:<15} → {action}')

## 📌 Summary

- **RouterResponse**: normalized return type — same shape regardless of which model ran
- **Routing saves cost**: simple tasks on mini, complex on flagship → 33-50% savings
- **Route by complexity** (token count, task type) = simplest effective strategy
- **Log every routing decision**: model chosen, reason, tokens — essential for debugging
- **Classify errors** to know the right fallback action: rate limit → wait/switch, timeout → retry